# Lesson 3 (Practice): EDA, sawal → chart → jawab

`09_charts_basics.ipynb` mein charts **banana** seekha. Is notebook mein unhein **istemal** karte hain: har hissa ek **cricket sawal** se shuru hota hai, phir chart, phir **jawab**.

Saath mein do nayi cheezein:
- `bins=5`: Matplotlib khud dabbe banaye to kya hota hai
- **IQR rule**: outliers ko formula se dhoondna (box plot andar se yahi karta hai)

Data: `innings_clean.csv` (12 innings, Lesson 2.4 mein saaf kiya hua)

In [ ]:
# Saare imports sab se upar
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv("../data/innings_clean.csv")
df.head(3)

---
## Sawal 1: Runs zyada tar kitne bante hain?
**Chart:** histogram

`bins=5` ka matlab: **"runs ki poori range (0 se 102) ko 5 barabar dabbon mein kaat do."**
Har dabba = 102 ÷ 5 = **20.4** runs chaura:
```
0–20.4 | 20.4–40.8 | 40.8–61.2 | 61.2–81.6 | 81.6–102
```

In [ ]:
plt.hist(df["runs"], bins=5, edgecolor="black")
plt.title("Runs distribution")
plt.xlabel("Runs")
plt.ylabel("Number of innings")
plt.show()

### Jawab
Ginti: **3, 3, 3, 1, 2**. Zyada tar innings (9 mein se 12) **0 se 61** ke beech, aur sirf 3 innings 61+.

### `bins=5` vs apni hadein
| | Faida | Nuksan |
|---|---|---|
| `bins=5` | jaldi, kisi bhi data pe chalta hai | hadein ajeeb (20.4, 40.8...) |
| `bins=[0, 20, 40, ...]` | cricket mein padhne layak (0–19, 20–39) | khud sochna padta hai |

Client ko chart dikhana ho to **apni saaf hadein** behtar hain.

---
## Sawal 2: Batsman zyada tar kitni balls khelta hai?

In [ ]:
plt.hist(df["balls"], bins=5, edgecolor="black")
plt.title("Balls distribution")
plt.xlabel("Balls")
plt.ylabel("Number of innings")
plt.show()

### Jawab
Ginti: **2, 2, 5, 1, 2**. Sab se lamba dabba **~30 se 43 balls**.

⚠️ Is dabbe mein **34 balls wali 2 innings** bhi hain, jo asal nahi, **median se bhari** thin. Yaani ye khamba thora "artificially" lamba hai. Imputation ka asar histogram pe bhi aata hai.

---
## Sawal 3: Kya zyada balls = zyada runs?
**Chart:** scatter (do numbers ka rishta)

In [ ]:
plt.scatter(df["balls"], df["runs"])
plt.title("Balls faced vs Runs")
plt.xlabel("Balls")
plt.ylabel("Runs")
plt.show()

### Jawab
1. Dots neeche-left se upar-right, **seedhi lakeer jaise** → haan, mazboot **linear** rishta. Linear Regression ke liye ideal shakal.
2. **x = 34 pe do dots** (20 aur 23 runs) ek ke upar ek aur lakeer se **neeche**: yahi median se bhari hui 2 rows hain.

⚠️ Lekin `balls` innings **ke baad** pata chalta hai. Innings se pehle prediction ke liye ye **leakage** hai.

---
## Sawal 4: Kis player ne sab se zyada runs banaye?
**Chart:** bar (groupby → `sort_values` → bar)

`sort_values(ascending=False)` lagane se khambe **bare se chhote** order mein aate hain. Chart padhna aur aasaan.

In [ ]:
totals = df.groupby("player")["runs"].sum().sort_values(ascending=False)
print(totals)

plt.bar(totals.index, totals.values)
plt.title("Total runs by player")
plt.xlabel("Player")
plt.ylabel("Total runs")
plt.show()

### Jawab
Fakhar **245**, Rizwan **152**, Babar **132**.
Lekin yaad rakhein: ye **total** hai. Babar ki innings sab se zyada (5) hain, phir bhi total kam: matlab us ka **average** bhi kam hoga (~26).

---
## Sawal 5: Kis team ke khilaf batting sab se aasaan thi?
**Chart:** bar (average) + countplot (kitni innings)

In [ ]:
avg_by_opposition = df.groupby("opposition")["runs"].mean().sort_values(ascending=False)
print(avg_by_opposition.round(1))

plt.bar(avg_by_opposition.index, avg_by_opposition.values)
plt.title("Average runs by opposition")
plt.xlabel("Opposition")
plt.ylabel("Average runs")
plt.show()

Yahi chart Seaborn se ek line mein (groupby woh khud karta hai, default **average**):

In [ ]:
sns.barplot(data=df, x="opposition", y="runs", errorbar=None)
plt.title("Average runs by opposition")
plt.show()

In [ ]:
sns.countplot(data=df, x="opposition")
plt.title("Innings per opposition")
plt.show()

### Jawab
Average mein **England** sab se upar (**53**), phir India (41.5), Australia (40.3).
Lekin countplot dekhein: England ke khilaf sirf **3 innings**. Itne kam data pe pakka faisla nahi kar sakte.

> Average dekhein to **count** hamesha saath dekhein.

---
## Sawal 6: Babar aur Fakhar ki form kaisi rahi?
**Chart:** line (filter → date se sort → plot)

In [ ]:
babar = df[df["player"] == "Babar"].sort_values("match_date")

plt.plot(babar["match_date"], babar["runs"], marker="o")
plt.title("Babar: runs over time")
plt.xlabel("Match date")
plt.ylabel("Runs")
plt.xticks(rotation=45)
plt.show()

In [ ]:
fakhar = df[df["player"] == "Fakhar"].sort_values("match_date")

plt.plot(fakhar["match_date"], fakhar["runs"], marker="o")
plt.title("Fakhar: runs over time")
plt.xlabel("Match date")
plt.ylabel("Runs")
plt.xticks(rotation=45)
plt.show()

### Jawab
- **Babar:** 45 → 12 → 0 → 20 → **55**. Beech mein kharab daur, aakhri match mein wapsi.
- **Fakhar:** 89 → 23 → 102 → 31. Upar neeche bohat: **inconsistent** lekin bare score ki salahiyat.

ML connection: Lesson 2.3 wala **"recent form"** feature (pichli innings ka average) isi chart ka number wala roop hai.

---
## Sawal 7: Kaunsa player kitna consistent hai?
**Chart:** box plot (har player ka)

In [ ]:
sns.boxplot(data=df, x="player", y="runs")
plt.title("Runs by player")
plt.show()

### Jawab
| Player | Kya dekhein | Matlab |
|---|---|---|
| Fakhar | median ~60, box **sab se lamba** | zyada runs, lekin unpredictable |
| Rizwan | median ~51, box **sab se chhota** | consistent anchor |
| Babar | median **20**, box neeche | is data mein form kamzor |

> **Lakeer kahan hai = kitne runs. Box kitna lamba hai = kitna bharosemand.**

---
## Sawal 8: Kya koi score "ajeeb" (outlier) hai? — IQR rule

Box plot ke bahar wale dots kis formula se bante hain? Yahi formula ab **khud** chalate hain.

```
Q1  = 25% wali value
Q3  = 75% wali value
IQR = Q3 − Q1                      (beech wali aadhi innings ki range)

lower = Q1 − 1.5 × IQR
upper = Q3 + 1.5 × IQR
Is range se bahar  →  outlier
```

| Line | Matlab |
|---|---|
| `quantile(0.25)` | 25% innings is se kam hain (Q1) |
| `quantile(0.75)` | 75% innings is se kam hain (Q3) |
| `df[(... < lower) \| (... > upper)]` | NumPy wali filtering: `\|` = "ya", har condition brackets mein |

In [ ]:
Q1 = df["runs"].quantile(0.25)
Q3 = df["runs"].quantile(0.75)
IQR = Q3 - Q1

lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

print(f"Q1: {Q1}, Q3: {Q3}, IQR: {IQR}")
print(f"Normal range: {lower} to {upper}")

df[(df["runs"] < lower) | (df["runs"] > upper)]

### Jawab (runs)
Range **−31.4 se 111.6** → koi row bahar nahi → **runs mein koi outlier nahi** (khali table).
Isi liye `Runs by player` box plot mein koi akela dot nahi tha.

### Ab yahi formula `balls` pe
Formula baar baar likhne ki bajaye ek **function** bana lete hain:

In [ ]:
def iqr_outliers(column):
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    print(f"{column}: normal range {lower:.1f} to {upper:.1f}")
    return df[(df[column] < lower) | (df[column] > upper)]

iqr_outliers("balls")

### Jawab (balls): 2 outliers mile!
- **3 balls** (Babar, duck)
- **70 balls** (Fakhar, 102 runs)

Ab **Inspect → Understand → Decide**: kya ye ghaltiyan hain?
**Nahi.** Cricket mein jaldi out hona (3 balls) aur lambi innings (70 balls) dono **bilkul asli** hain. Inhein hatana ghalat hoga, ye model ke liye qeemti information hai.

Mukabla karein: Lesson 2.4 ka **610 runs** aur **−12 balls** na-mumkin thay → woh **ghaltiyan** thin.

> **Outlier ≠ ghalti.** Formula sirf **shak** paida karta hai. Faisla **cricket ki samajh** karti hai.

In [ ]:
# Box plot pe bhi wahi 2 dots nazar aayenge
sns.boxplot(data=df, y="balls")
plt.title("Balls: outliers")
plt.show()

---
## Sawal 9: Kaunse numeric columns aapas mein jure hain?
**Chart:** correlation heatmap

In [ ]:
corr = df.corr(numeric_only=True)
corr.round(2)

In [ ]:
sns.heatmap(corr, annot=True, cmap="coolwarm", vmin=-1, vmax=1)
plt.title("Correlation")
plt.show()

### Jawab
| Jora | Number | Matlab |
|---|---|---|
| balls ↔ runs | **0.95** | bohat mazboot, lekin 🚨 **leakage** (innings ke baad pata) |
| batting_position ↔ runs | **−0.38** | bharosa nahi: position **category** hai, aur har position pe sirf ek player hai, to ye asal mein **Babar ki form** naap raha hai |
| batting_position ↔ balls | **−0.41** | wahi wajah |

> **Correlation ≠ causation.** Peeche koi teesri cheez (yahan **player**) chhupi ho sakti hai.

---
## Khulasa: EDA ka tareeqa

```
1. Sawal likho (cricket ki zubaan mein)
2. Sahi chart chuno (hist / scatter / bar / line / box / heatmap)
3. Chart padh ke jawab likho
4. Shak karo: count kitna hai? imputed values? leakage? chhupi wajah?
5. Outlier mile → Inspect → Understand → Decide (hatana zaroori nahi)
```